# 03 — Sparse Retrieval and BM25

**Repo:** ai-learning-notes | **Folder:** rag-and-retrieval  
**Built alongside:** local-rag-llm (github.com/RT91-data/local-rag-llm)

---

## What this notebook covers

- Why sparse retrieval exists alongside dense retrieval
- TF-IDF — the foundation BM25 is built on
- The BM25 formula — every parameter explained
- When BM25 beats dense retrieval and when it loses
- The k1 and b tuning parameters
- Why BM25 is critical for D365 and ERP document RAG
- Failure modes and limitations

**Pure Python only — no external dependencies.**

---
## 1. The Problem Dense Retrieval Cannot Solve

Dense retrieval (FAISS + embeddings) works by mapping text to a vector in semantic space.  
Two pieces of text with similar *meaning* end up near each other, even if they share no words.

**This is a strength — and a weakness.**

Consider a user searching for a specific D365 vendor account:  
`"What are the payment terms for vendor CONTOSO-001?"`

The embedding model has no special knowledge of `CONTOSO-001`.  
It treats it as an unknown token sequence and maps it somewhere in embedding space.  
There is no guarantee that the chunk containing `CONTOSO-001` ends up semantically close to the query.

BM25 has no such problem. It looks for the literal string `CONTOSO-001` in chunks.  
If a chunk contains it, the chunk scores high. Simple, reliable, zero ambiguity.

**The rule:**  
Dense retrieval excels at semantic matching — concepts, paraphrases, intent.  
Sparse retrieval (BM25) excels at exact matching — codes, IDs, names, technical terms, rare words.

In [ ]:
# Illustrating where dense retrieval fails on exact terms

chunks = [
    "CONTOSO-001 has payment terms NET30 and a credit limit of SGD 50,000.",
    "FABRIKAM-002 has payment terms NET60 and a credit limit of SGD 25,000.",
    "Vendor payment terms define when invoices must be settled after receipt.",
    "Credit limits control the maximum outstanding balance per vendor account.",
]

# Simulated dense similarity scores for query "payment terms for CONTOSO-001"
# Dense model understands 'payment terms' semantically but not the vendor ID
dense_scores = [0.71, 0.68, 0.75, 0.69]  # note: chunk 0 NOT highest — ID not recognised

# BM25 exact keyword match for same query
def count_keyword_matches(query, chunk):
    query_words = set(query.lower().split())
    chunk_words = chunk.lower().split()
    return sum(1 for w in chunk_words if w.rstrip('.,') in query_words)

query = "payment terms for CONTOSO-001"
bm25_scores = [count_keyword_matches(query, c) for c in chunks]

print(f"Query: '{query}'")
print()
print(f"{'#':<3} {'Dense score':>12} {'BM25 score':>12}  Chunk")
print("-" * 75)
for i, (chunk, ds, bs) in enumerate(zip(chunks, dense_scores, bm25_scores)):
    print(f"{i:<3} {ds:>12.3f} {bs:>12}  {chunk[:60]}...")

print()
dense_winner = dense_scores.index(max(dense_scores))
bm25_winner  = bm25_scores.index(max(bm25_scores))
print(f"Dense retrieval returns chunk {dense_winner}: '{chunks[dense_winner][:50]}...'")
print(f"BM25 retrieval returns chunk {bm25_winner}:  '{chunks[bm25_winner][:50]}...'")
print()
print("Dense missed CONTOSO-001 because the ID has no semantic meaning to the model.")
print("BM25 found it by exact string matching — the right answer every time.")

---
## 2. TF-IDF — The Foundation

BM25 is a refinement of TF-IDF. Understanding TF-IDF first makes BM25 obvious.

**TF — Term Frequency**  
How many times does the query term appear in the chunk?  
More occurrences → higher score.  
`TF(term, chunk) = count of term in chunk`

**IDF — Inverse Document Frequency**  
How rare is the query term across all chunks?  
Rarer terms are more informative.  
"the" appears everywhere → low IDF (not useful for retrieval).  
"CONTOSO-001" appears in one chunk → high IDF (very useful for retrieval).  
`IDF(term) = log( N / df(term) )`  
where N = total chunks, df = number of chunks containing the term

**TF-IDF score**  
`score(term, chunk) = TF × IDF`

**Problem with raw TF:**  
A chunk that mentions a term 100 times isn't 100× more relevant than one mentioning it once.  
Relevance increases with frequency but levels off.  
BM25 fixes this with a saturation function.

In [ ]:
import math

corpus = [
    "Zero Ambient Authority means agents must not inherit full admin privileges.",
    "JIT downscoping provides task-scoped credentials that expire immediately after the task.",
    "Agents must never be granted a global key or broad standing permissions.",
    "The Confused Deputy problem occurs when prompt injection tricks an agent.",
    "SPIFFE IDs provide unique cryptographic identity to every agent in the system.",
]

def tokenise(text):
    """Simple whitespace tokeniser with lowercasing."""
    return [w.lower().rstrip('.,!?;:') for w in text.split()]

def compute_tf(term, chunk_tokens):
    return chunk_tokens.count(term)

def compute_idf(term, corpus_tokens):
    N = len(corpus_tokens)
    df = sum(1 for doc in corpus_tokens if term in doc)
    if df == 0:
        return 0
    return math.log((N - df + 0.5) / (df + 0.5) + 1)  # smooth IDF (BM25 variant)

corpus_tokens = [tokenise(doc) for doc in corpus]

# Show IDF values for different terms
terms_to_check = ["agent", "agents", "the", "credentials", "spiffe", "privileges"]

print("IDF VALUES — higher means rarer and more informative")
print("=" * 50)
print(f"{'Term':<15} {'Doc freq':>10} {'IDF':>10}")
print("-" * 40)
for term in terms_to_check:
    df = sum(1 for doc in corpus_tokens if term in doc)
    idf = compute_idf(term, corpus_tokens)
    print(f"{term:<15} {df:>10} {idf:>10.4f}")

print()
print("'spiffe' appears in 1 doc → high IDF (rare, very informative)")
print("'agent/agents' appears in most docs → low IDF (common, less discriminating)")
print("'the' would have IDF ≈ 0 in a large corpus (appears everywhere)")

---
## 3. The BM25 Formula

BM25 (Best Match 25) adds two improvements over raw TF-IDF:

**1. Term frequency saturation (k1 parameter)**  
Raw TF grows linearly — a term appearing 10 times scores 10× a term appearing once.  
BM25 applies a saturation function so the score levels off:
```
saturated_TF = (tf × (k1 + 1)) / (tf + k1)
```
At k1=1.5: tf=1 → 1.0, tf=5 → 1.45, tf=10 → 1.58, tf=100 → 1.64  
The gain from tf=1 to tf=5 is much bigger than from tf=50 to tf=100.

**2. Document length normalisation (b parameter)**  
A long chunk has more words, so it will naturally contain more term occurrences.  
BM25 penalises long chunks to level the playing field:
```
length_norm = 1 - b + b × (chunk_length / avg_chunk_length)
```
b=0: no length normalisation.  
b=1: full normalisation — only term density matters, not absolute count.

**Full BM25 score for a query term:**
```
score = IDF × (tf × (k1 + 1)) / (tf + k1 × length_norm)
```
For a multi-term query: sum the scores across all query terms.

In [ ]:
class BM25:
    """
    BM25 implementation from scratch.
    Matches the algorithm used by rank_bm25 (which LangChain uses internally).
    """

    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b  = b
        self.corpus_tokens = [tokenise(doc) for doc in corpus]
        self.corpus = corpus
        self.N = len(corpus)
        self.avgdl = sum(len(d) for d in self.corpus_tokens) / self.N

    def idf(self, term):
        df = sum(1 for doc in self.corpus_tokens if term in doc)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def score(self, query, doc_idx):
        query_terms = tokenise(query)
        doc_tokens  = self.corpus_tokens[doc_idx]
        dl = len(doc_tokens)
        total = 0.0
        for term in query_terms:
            tf   = doc_tokens.count(term)
            idf  = self.idf(term)
            norm = 1 - self.b + self.b * (dl / self.avgdl)
            total += idf * (tf * (self.k1 + 1)) / (tf + self.k1 * norm)
        return total

    def retrieve(self, query, k=3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x: x[1], reverse=True)[:k]


corpus = [
    "Zero Ambient Authority means agents must never inherit full admin privileges from their parent.",
    "Just-In-Time downscoping provides hyper-restricted credentials scoped to the exact task.",
    "The Confused Deputy problem occurs when prompt injection tricks an over-privileged agent.",
    "SPIFFE IDs provide unique cryptographic identity to every individual agent in the system.",
    "Ephemeral sandboxes reset their state completely between each vibe loop iteration.",
    "The Vibe Diff translates complex generated code into plain English before approval.",
]

bm25 = BM25(corpus)
query = "how are agent credentials scoped and restricted?"

print(f"Query: '{query}'")
print()
print(f"{'Rank':<5} {'Score':>8}  Chunk")
print("-" * 75)
results = bm25.retrieve(query, k=3)
for rank, (idx, score) in enumerate(results, 1):
    print(f"{rank:<5} {score:>8.4f}  {corpus[idx][:65]}...")

print()
print("BM25 correctly surfaces credential-related chunks via keyword matching.")

---
## 4. The k1 and b Parameters — What They Control

BM25 has two tuning parameters. Understanding them matters for D365/ERP document RAG.

In [ ]:
# Visualising term frequency saturation at different k1 values

def saturated_tf(tf, k1):
    """How BM25 dampens raw term frequency."""
    return (tf * (k1 + 1)) / (tf + k1)

tf_values = [1, 2, 3, 5, 10, 20, 50]
k1_values = [0.5, 1.2, 1.5, 2.0]  # LangChain BM25Retriever default is 1.5

print("TF SATURATION — how k1 controls diminishing returns")
print("Higher k1 = more reward for repeated terms (less saturation)")
print("Lower k1  = faster saturation (1 occurrence ≈ 10 occurrences)")
print()
print(f"{'Raw TF':>8}", end="")
for k1 in k1_values:
    print(f"  k1={k1:>3}", end="")
print()
print("-" * 45)
for tf in tf_values:
    print(f"{tf:>8}", end="")
    for k1 in k1_values:
        print(f"  {saturated_tf(tf, k1):>6.3f}", end="")
    print()

print()
print("LangChain BM25Retriever uses k1=1.5 (default).")
print("For ERP IDs that appear exactly once per chunk, k1 value makes little difference.")
print("For narrative text where repetition signals importance, tune k1 higher (1.8-2.0).")
print()
print("b PARAMETER — document length normalisation")
print("-" * 50)
print("b=0.0: ignore chunk length — absolute term count matters")
print("b=0.75: standard default — partial normalisation")
print("b=1.0: full normalisation — only term density matters, not chunk length")
print()
print("For RAG with consistent chunk sizes (RecursiveCharacterTextSplitter):")
print("b=0.75 is fine — chunks are similar length so normalisation has small effect.")

---
## 5. When BM25 Wins vs When Dense Wins

This is the core decision for understanding why hybrid retrieval is necessary.

In [ ]:
# Head-to-head comparison across different query types

scenarios = [
    {
        "query_type": "Exact ID lookup",
        "example_query": "CONTOSO-001 payment terms",
        "bm25_wins": True,
        "reason": "CONTOSO-001 is an unknown token to embedding model — BM25 finds it exactly",
        "erp_relevance": "HIGH — vendor IDs, account numbers, journal entries",
    },
    {
        "query_type": "Technical acronym",
        "example_query": "SPIFFE ID assignment",
        "bm25_wins": True,
        "reason": "SPIFFE is rare — high IDF, BM25 surfaces it precisely",
        "erp_relevance": "HIGH — D365 module codes, configuration keys",
    },
    {
        "query_type": "Semantic paraphrase",
        "example_query": "how do agents get temporary credentials?",
        "bm25_wins": False,
        "reason": "Answer uses 'JIT downscoping' — no word overlap with query",
        "erp_relevance": "HIGH — 'approve vendor payment' vs 'authorise outbound transfer'",
    },
    {
        "query_type": "Conceptual question",
        "example_query": "what prevents agents from gaining too much access?",
        "bm25_wins": False,
        "reason": "Semantic match needed — answer spans ZAA, JIT, ABAC concepts",
        "erp_relevance": "HIGH — 'approval workflows' vs 'authorisation controls'",
    },
    {
        "query_type": "Numeric value",
        "example_query": "credit limit 50000",
        "bm25_wins": True,
        "reason": "Numbers are exact — embeddings don't understand 50000 > 25000",
        "erp_relevance": "HIGH — invoice amounts, budget codes, GL account numbers",
    },
    {
        "query_type": "Cross-language / synonym",
        "example_query": "container isolation for code execution",
        "bm25_wins": False,
        "reason": "Answer uses 'ephemeral sandbox' and 'gVisor' — different words entirely",
        "erp_relevance": "MEDIUM — 'cost centre' vs 'profit centre' queries",
    },
]

print("BM25 vs DENSE RETRIEVAL — when each wins")
print("=" * 70)
for s in scenarios:
    winner = "🔤 BM25 wins" if s['bm25_wins'] else "🧠 Dense wins"
    print(f"\n{s['query_type']} — {winner}")
    print(f"  Query:  '{s['example_query']}'")
    print(f"  Why:    {s['reason']}")
    print(f"  D365:   {s['erp_relevance']}")

print()
print("="*70)
bm25_wins = sum(1 for s in scenarios if s['bm25_wins'])
dense_wins = len(scenarios) - bm25_wins
print(f"BM25 wins: {bm25_wins}/{len(scenarios)} scenarios")
print(f"Dense wins: {dense_wins}/{len(scenarios)} scenarios")
print("Neither alone is sufficient. Hybrid retrieval covers both.")

---
## 6. Why BM25 Is Especially Important for D365 / ERP RAG

Enterprise ERP documents are full of exact-match requirements that dense retrieval handles poorly:

| Content type | Example | Why BM25 critical |
|---|---|---|
| **Vendor accounts** | CONTOSO-001, FABRIKAM-002 | IDs unknown to embedding model |
| **GL account codes** | 110000, 600100, 900000 | Numbers lack semantic embedding |
| **D365 module names** | SalesTable, VendTable, LedgerJournalTrans | Technical names = rare tokens |
| **Configuration keys** | SysConfig.EnableWorkflow | Exact strings only |
| **Transaction numbers** | INV-2026-04521, PO-2026-08823 | Unique IDs, no semantic meaning |
| **Error codes** | ERR_POSTING_BLOCKED, AXERRP001 | Rare, exact, critical |
| **Amounts** | 45000.00 SGD, USD 1,250,000 | Numbers are poor in embedding space |
| **Dates** | 2026-06-30, FY2026 Q2 | Temporal values lose meaning in vectors |

A RAG system built on pure dense retrieval for D365 would fail on a large fraction of real user queries.  
BM25 is not optional for enterprise ERP RAG — it is mandatory.

In [ ]:
# Simulating D365 ERP document retrieval

d365_chunks = [
    "VendTable record CONTOSO-001: PaymTermId=NET30, CreditMax=50000.00, CurrencyCode=SGD, Blocked=No.",
    "VendTable record FABRIKAM-002: PaymTermId=NET60, CreditMax=25000.00, CurrencyCode=USD, Blocked=No.",
    "LedgerJournalTrans INV-2026-04521: AccountNum=200100, Amount=12500.00, TransDate=2026-06-15.",
    "SysConfig key EnableApprovalWorkflow: Value=True. Requires minimum approvers=2 for amounts over 10000.",
    "ERR_POSTING_BLOCKED: Posting blocked because vendor CONTOSO-001 has exceeded credit limit.",
    "Payment terms NET30 mean the invoice must be settled within 30 days of the invoice date.",
]

bm25_d365 = BM25(d365_chunks)

queries = [
    "What is the credit limit for CONTOSO-001?",
    "Why is posting blocked for CONTOSO-001?",
    "Show me transaction INV-2026-04521",
    "What does NET30 mean for payment?",
]

for query in queries:
    results = bm25_d365.retrieve(query, k=1)
    top_idx, top_score = results[0]
    print(f"Q: {query}")
    print(f"   BM25 top result (score={top_score:.3f}):")
    print(f"   '{d365_chunks[top_idx][:75]}...'")
    print()

---
## 7. BM25 in LangChain — How local-rag-llm Uses It

LangChain's `BM25Retriever` wraps the `rank_bm25` library.  
It's used in local-rag-llm as one half of the `EnsembleRetriever`.

```python
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Build BM25 from all chunks
bm25_ret = BM25Retriever.from_documents(clean_chunks)
bm25_ret.k = 4   # return top 4 results

# FAISS retriever
faiss_ret = vectorstore.as_retriever(search_kwargs={"k": 4})

# Combine: 40% BM25, 60% FAISS
ensemble = EnsembleRetriever(
    retrievers=[bm25_ret, faiss_ret],
    weights=[0.4, 0.6]
)
```

**Important limitation:** `BM25Retriever` has no `save_local()` method.  
The BM25 index is rebuilt from source PDFs on every app restart.  
For 250 chunks this takes ~2 seconds — acceptable for local deployment.  
For 50,000 chunks this would take minutes — use Elasticsearch instead.

**The 40/60 weighting:**  
This is a default, not an empirically tuned value.  
The correct approach is to run RAGAS evaluation across weight combinations  
and pick the one with the best context recall across your specific document corpus.

---
## 8. BM25 Failure Modes

BM25 is powerful but has well-known failure modes:

In [ ]:
# BM25 failure modes with examples

failure_modes = [
    {
        "name": "Vocabulary mismatch",
        "query": "How do we stop agents from acting autonomously?",
        "answer_contains": "Zero Ambient Authority, JIT downscoping, ABAC",
        "problem": "No word overlap between query and answer — BM25 scores near zero",
        "fix": "Dense retrieval covers this — why hybrid is essential",
    },
    {
        "name": "Stop word dominance",
        "query": "What is the role of the agent?",
        "answer_contains": "Agent role description",
        "problem": "'what', 'is', 'the', 'of' are stop words → low IDF. 'agent' is everywhere → also low IDF",
        "fix": "Remove stop words in tokenisation (LangChain BM25Retriever does this)",
    },
    {
        "name": "Morphological variants",
        "query": "agents credentials expire",
        "answer_contains": "agent credential expiry",
        "problem": "'agents' ≠ 'agent', 'expire' ≠ 'expiry' — exact string mismatch",
        "fix": "Stemming or lemmatisation before indexing (not in LangChain default)",
    },
    {
        "name": "Term frequency spam",
        "query": "security pillar",
        "answer_contains": "Specific pillar description",
        "problem": "A chunk mentioning 'security' 20 times gets high score even if irrelevant",
        "fix": "k1 saturation limits this — BM25 advantage over raw TF",
    },
    {
        "name": "No semantic understanding",
        "query": "container that resets between runs",
        "answer_contains": "ephemeral sandbox",
        "problem": "'container' and 'ephemeral sandbox' have zero word overlap",
        "fix": "Dense retrieval only solution — BM25 simply cannot handle this",
    },
]

print("BM25 FAILURE MODES")
print("=" * 70)
for i, fm in enumerate(failure_modes, 1):
    print(f"\n{i}. {fm['name']}")
    print(f"   Query:   '{fm['query']}'")
    print(f"   Answer:  {fm['answer_contains']}")
    print(f"   Problem: {fm['problem']}")
    print(f"   Fix:     {fm['fix']}")

---
## 9. Query Rewriting as a BM25 Enhancer

When a user asks a conversational follow-up — "What about the second one?" — both BM25 and dense retrieval fail.  
There are no useful keywords, and the semantic meaning is entirely context-dependent.

local-rag-llm solves this with **query rewriting** before retrieval:  
"What about the second one?" → "What is Pillar 2 - Data security in the 7-pillar agent security architecture?"

The rewritten query contains the right keywords for BM25 AND the right semantic content for FAISS.  
Both retrievers perform better on the expanded query than on the original.

```python
# From local-rag-llm app.py
def rewrite_query_for_retrieval(query: str, conversation_history: list) -> str:
    # Uses Claude Haiku to expand conversational follow-ups
    # into standalone, keyword-rich search queries
    # Returns original query unchanged if already standalone
    ...
```

This is why query rewriting is placed BEFORE retrieval in the pipeline, not after.

In [ ]:
# Demonstrating query rewriting impact on BM25

corpus = [
    "Pillar 1 - Infrastructure & Networking: isolate runtime in ephemeral kernel-level sandboxes.",
    "Pillar 2 - Data: protect context with CMEK and mTLS. Enforce tenant partitioning.",
    "Pillar 3 - Model: treat system instructions as cryptographically attested artifacts.",
    "Pillar 4 - Application & Runtime: deploy LLM firewalls and Centralised Agent Gateways.",
    "Pillar 5 - IAM: SPIFFE IDs, ABAC, JIT downscoping. Intent × User × Time matrix.",
]

bm25 = BM25(corpus)

original_query  = "What about the second one?"
rewritten_query = "What is Pillar 2 Data security in the agent security architecture?"

print("QUERY REWRITING — impact on BM25 retrieval")
print("=" * 65)

for label, query in [("Original", original_query), ("Rewritten", rewritten_query)]:
    results = bm25.retrieve(query, k=2)
    print(f"\n{label} query: '{query}'")
    print(f"{'Rank':<5} {'Score':>8}  Chunk")
    print("-" * 65)
    for rank, (idx, score) in enumerate(results, 1):
        print(f"{rank:<5} {score:>8.4f}  {corpus[idx][:60]}...")

print()
print("Original query has no useful keywords → BM25 retrieves arbitrarily.")
print("Rewritten query contains 'Pillar 2', 'Data' → BM25 immediately finds the right chunk.")

---
## 10. Summary

| Concept | Key point |
|---|---|
| **Sparse retrieval** | Keyword-based — finds exact matches that dense misses |
| **TF** | Raw term count — how many times a term appears in a chunk |
| **IDF** | Rarity score — rare terms are more informative than common ones |
| **BM25** | TF-IDF with saturation (k1) and length normalisation (b) |
| **k1=1.5** | Standard default — moderate saturation for most document types |
| **b=0.75** | Standard default — partial length normalisation |
| **BM25 wins** | Exact IDs, codes, amounts, rare technical terms, acronyms |
| **Dense wins** | Paraphrases, synonyms, conceptual questions, cross-language |
| **D365 relevance** | BM25 is critical — vendor IDs, GL codes, transaction numbers are all exact-match |
| **No save_local** | BM25 must be rebuilt on every startup — fine at <10K chunks |
| **Query rewriting** | Expands ambiguous queries into keyword-rich forms — boosts BM25 recall |

**The single most important insight from this notebook:**  
For enterprise ERP document RAG, BM25 is not optional.  
Vendor IDs, account codes, transaction numbers, and configuration keys are invisible to embedding models.  
BM25 finds them reliably. Dense retrieval finds meaning. You need both.

---
## Next Notebooks

- **04** — Hybrid retrieval (EnsembleRetriever internals, weighting, Reciprocal Rank Fusion)
- **05** — Reranking with CrossEncoders (bi-encoder vs cross-encoder, ms-marco internals)
- **06** — RAG evaluation with RAGAS (metrics, golden dataset, baselines from real eval)